# Instructions for this notebook:

Prior to running this notebook, the following data must be downloaded and installed in the project folder under Data/Raw/manning/:

[CLRH Hydrofabric dataset](https://hydrology.uwaterloo.ca/CLRH/Hydrofabric.html) (download watershed of interest based on outlet point) 
- Extract zip file to: /Data/Raw/manning/WB_Data/

[NHN streams dataset](https://ftp.maps.canada.ca/pub/nrcan_rncan/vector/geobase_nhn_rhn/shp_en/) (download shapefile corresponding to outlet point ID (e.g., 05OE000)) 
- Extract zip file to: /Data/Raw/manning/WB_Data/

[LiDAR DEM](https://mli.gov.mb.ca/dems/index_external_lidar.html) (download watershed of interest) ***CLIPPED AND RESAMPLED IN ARCGIS PRO
- Extract zip file to: /Data/Raw/manning/WB_Data/

# Calculate Depression Depths (mm) per Subbasin

In [1]:
# Sytem packages
import os
import sys
import fileinput


# Analysis packages
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString, Polygon, MultiPoint, mapping
from shapely.geometry import shape as shapely_shape
import rasterio
from rasterio.crs import CRS
from rasterio.mask import mask
from bs4 import BeautifulSoup


# Visualization packages
import matplotlib.pyplot as plt
import rasterio.plot


sys.path.append("..")
from src.py.proj_conf import *

## Process DEM and Stream Network

### Project subbasins shapefile to align with LiDAR raster DEM

In [2]:
# Load CLRH subbasins shapefile
clrh_gdf_file = DATA_RAW + "WB_Data/finalcat_info_v1-0.shp"
clrh_gdf = gpd.read_file(clrh_gdf_file)

In [ ]:
# LiDAR DEM file name with path
input_DEM_file = DATA_RAW + "WB_Data/sr_dem_cgvd28.tif"


# Open LiDAR DEM and check CRS
with rasterio.open(input_DEM_file) as src:
    input_DEM_crs = src.crs


# Project subbasins data to match DEM
clrh_gdf_projected = clrh_gdf.to_crs(input_DEM_crs)


# Remove any non-alphanumeric characters from crs name
input_DEM_crs_alnum = ''.join(c for c in \
                              str(input_DEM_crs) if c.isalnum())


# Projected subbasins shapefile name with path
clrh_gdf_projected_file = DATA_RAW + \
                          f"WB_Data/clrh_basins_projected_{input_DEM_crs_alnum}.shp"


# Write projected subbasins data to shapefile
clrh_gdf_projected.to_file(clrh_gdf_projected_file)


# Check if shapefile projection aligns with DEM projection
is_correctly_projected = (input_DEM_crs == clrh_gdf_projected.crs)
                          

# Print results
print("Shapefile projection is aligned with DEM projection: ", 
      is_correctly_projected)
print(f"Projection is: {input_DEM_crs}")
print("\nIf True and EPSG:26914 are returned, proceed to clipping.")

### Clip LiDAR raster DEM to watershed

In [ ]:
# Convert projected subbasins data to GeoJSON-like format
shapes = [mapping(geom) for geom in clrh_gdf_projected.geometry]


# Mask (clip) the input DEM file
with rasterio.open(input_DEM_file) as src:
    nodata_value = src.nodata
    out_image, out_transform = mask(src, 
                                    shapes, 
                                    crop=True,
                                    all_touched=True                  
                                    )
    out_meta = src.meta.copy()


# Update the copied metadata to match the clipped raster's dimensions and transform
out_meta.update({
    "driver": "GTiff",
    "height": out_image.shape[1],
    "width": out_image.shape[2],
    "transform": out_transform
})


# Clipped DEM file name with path
output_DEM_file = DATA_RAW + "WB_Data/sr_dem_cgvd28_Clip.tif"


# Write clipped DEM data into file
with rasterio.open(output_DEM_file,
                   "w", 
                   **out_meta) as dest:
    dest.write(out_image)


# Print out some stats fyi
valid_out_image = out_image[0][out_image[0] != nodata_value]
print("Clipped unique values:", np.unique(valid_out_image))
print("Clipped number of unique values: ", len(np.unique(valid_out_image)))
print("Clipped maximum value :", np.max(valid_out_image))
print("Clipped minimum value (excluding nodata value): ",  np.min(valid_out_image))
print("\nIf the numbers make sense, proceed to resampling.")


""" #=======================ELENA CODE (WILL DELETE IT SOON)===================================
### SUBBASIN AND STREAM SHAPEFILES WERE PROJECTED IN ARCGIS PRO (NO PROJECTION TOOL IN WHITEBOX)

# Clip NHN streams to study area
# wbt.clip(
#    i = 'NHN_05OE000_5_0_HD_SLWATER_1.shp',
#    clip = 'clrh_basins_projected_4617.shp',
#    output = 'NHN_Clip.shp'
#)

### RASTER DATA WAS CLIPPED AND RESAMPLED IN ARCGIS PRO DUE TO MEMORY ALLOCATION ERROR

# Clip dem raster to study area
#wbt.clip_raster_to_polygon(
    #i = 'sr_dem_cgvd28.tif',
    #polygons = 'finalcat_info_v1-0_Project.shp',
    #output = 'sr_dem_cgvd28_Clip_MC.tif'
#)

# Resample DEM to 5m (for faster processing)
#wbt.resample(
    #inputs = 'sr_dem_cgvd28_Clip_MC.tif',
    #output = 'sr_dem_cgvd28_Clip_MC_Resample_5m_corr',
    #cell_size = 5,
    #method = "cc"
#) """

### Resample clipped LiDAR raster DEM to 5m resolution

In [ ]:
# Import whitebox
from WBT.whitebox_tools import WhiteboxTools

# Save current working directory
default_directory = os.getcwd()

wbt = WhiteboxTools()

# This line could probably could get cleaned up...
wbt.set_working_dir(os.path.abspath(os.path.join(os.path.dirname("__file__"),  '..', 'Data/Raw/manning/WB_Data')))

In [ ]:
# Define resolution
resolution = 5 
resolution_units = "m" # metres


# Resample to defined resolution
wbt.resample(
    inputs="sr_dem_cgvd28_Clip.tif",
    output=f"sr_dem_cgvd28_Clip_Resample_{resolution}{resolution_units}",
    cell_size = resolution,
    method = "cc"
)


# After using whitebox, jump back into default directory to avoid path errors
os.chdir(default_directory)


# Clipped resampled DEM file name with path
output_DEM_clires_file = DATA_RAW + \
                        f"WB_Data/sr_dem_cgvd28_Clip_Resample_{resolution}{resolution_units}.tif"


# Open the clipped resampled file to get some stats
with rasterio.open(output_DEM_clires_file) as src:
    image = src.read(1)


# Print out some stats fyi
valid_image = np.ma.masked_equal(image, nodata_value)
print("Clipped resampled DEM unique values:", np.unique(valid_image))
print("Clipped resampled number of unique values: ", len(np.unique(valid_image)))
print("Clipped resampled maximum value :", np.max(valid_image))
print("Clipped resampled minimum value (excluding nodata value): ",  np.min(valid_image))
print("\nIf the numbers make sense, proceed to next sections.")


### Visualize clipped and resampled LiDAR raster DEM

In [ ]:
# Open the clipped and resampled file to visualize
with rasterio.open(output_DEM_clires_file) as src:
    image = src.read(1)
    extent = rasterio.plot.plotting_extent(src)


# Visualize clipped and resampled file
valid_image = np.ma.masked_equal(image, nodata_value)
plt.figure(figsize=(10, 8))
plt.imshow(valid_image, cmap='terrain', extent=extent)
plt.colorbar(label='Elevation (m)')
plt.title("Clipped & Resampled LiDAR DEM")
plt.xlabel("Easting")
plt.ylabel("Northing")
plt.show()

### ANYTHING BEYOND THIS POINT IS BROKEN CODE (IY NEEDS TO CLEAN IT UP)

In [ ]:
os.getcwd()

### RUNNING WHITEBOX TOOLS CHANGES THE WORKING DIRECTORY (ADDS WBT FOLDER). NOTEBOOK MUST BE RUN TOP TO BOTTOM, OTHERWISE PATHS WITLL NOT WORK

In [19]:
# Project NHN clipped shapefile
streams_clip = gpd.read_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + "NHN_Clip.shp")
streams_clip_proj_26914 = streams_clip.to_crs(26914)
streams_clip_proj_26914.to_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + "NHN_Clip_Project_26914.shp")

## Generate Depressions Raster

In [ ]:
# Burn streams into DEM and fill
wbt.fill_burn(
    dem = 'sr_dem_cgvd28_Clip_MC_Resample_5m_corr.tif',
    streams = 'NHN_Clip_Project_26914.shp',
    output = 'DEM_Clip_MC_Resample_5m_FillBurn.tif'
)

# Subtract raw DEM from filled DEM
wbt.raster_calculator(
    output = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps.tif',
    statement = "'DEM_Clip_MC_Resample_5m_FillBurn.tif' - 'sr_dem_cgvd28_Clip_MC_Resample_5m_corr.tif'"
)

# Remove stray burn lines
wbt.conditional_evaluation(
    i = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps.tif',
    output = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps_Corr.tif',
    statement = "value < 0.0",
    true = 0.0,
    false = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps.tif'
)

## Calculate Depression Depths per Subbasin

In [ ]:
# Convert clrh subbasin polygons to raster
wbt.vector_polygons_to_raster(
    i = 'clrh_basins_projected_26914.shp',
    output = 'clrh_basins_raster.tif',
    field = "FID",
    nodata = True,
    cell_size = 5.0
)

# Define projection of clrh raster
with rasterio.open(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + 'clrh_basins_raster.tif', 'r+') as rds:
    rds.crs = CRS.from_epsg(26914)

# Calculate zonal statistics
wbt.zonal_statistics(
    i = 'DEM_Clip_MC_Resample_5m_FillBurn_Deps_Corr.tif',
    features = 'clrh_basins_raster.tif',
    stat="total",
    out_table='ZonalStats_DEM_Clip_MC_Resample_5m_FillBurn_Deps_Corr.html'
)

In [13]:
# Read html file as pandas dataframe
file = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + 'ZonalStats_DEM_Clip_MC_Resample_5m_FillBurn_Deps_Corr.html'
zonal_stats_dfs = pd.read_html(file, flavor='bs4')
zonal_stats_df = zonal_stats_dfs[0]

# Join totals to clrh subbasins geodataframe
clrh_gdf_projected_26914['Deps_SUM'] = zonal_stats_df['Total']

# Calculate depression volumes (m3) and add field to gdf
clrh_gdf_projected_26914['Deps_Vol_m3'] = clrh_gdf_projected_26914['Deps_SUM'] * 5 * 5

# Calculate depression depths (mm) and add field to gdf
clrh_gdf_projected_26914['Deps_Depth_mm'] = clrh_gdf_projected_26914['Deps_Vol_m3'] / clrh_gdf_projected_26914['BasArea'] * 1000

In [ ]:
# Write geodataframe to file for use in raven input file creation
clrh_gdf_projected_26914.to_file(os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', '..', 'Data/Raw/manning/WB_Data')) + "/" + "clrh_basins_deps.shp")
